In [1]:
import os
import joblib
import pandas as pd
import xgboost as xgb
from sklearn import metrics
from sklearn.model_selection import GridSearchCV, train_test_split

MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)

# ============================================
# 1. RP model
# ============================================
datasets_rp = pd.read_excel('preparation - rp.xlsx')
X_rp = datasets_rp.iloc[:, 2:5].to_numpy()
y_rp = datasets_rp.iloc[:, 5]

X_train, X_test, y_train, y_test = train_test_split(
    X_rp, y_rp, test_size=0.2, random_state=108
)
model_rp = xgb.XGBRegressor(n_estimators=300)

param_grid_rp = {
    'colsample_bytree': [0.77],
    'learning_rate':    [0.218],
    'max_depth':        [7],
    'alpha':            [0.03]
}
grid_rp = GridSearchCV(model_rp, param_grid_rp, scoring='r2', cv=5, n_jobs=-1)
grid_rp.fit(X_train, y_train)
best_rp = grid_rp.best_estimator_

print("RP Train R2:", metrics.r2_score(y_train, best_rp.predict(X_train)))
print("RP Test  R2:", metrics.r2_score(y_test,  best_rp.predict(X_test)))

joblib.dump(best_rp, os.path.join(MODEL_DIR, 'best_rp.pkl'))

# ============================================
# 2. SJ model
# ============================================
datasets_sj = pd.read_excel('preparation - D.xlsx')
X_sj = datasets_sj.iloc[:, 0:5].to_numpy()
y_sj = datasets_sj.iloc[:, 5]

X_train, X_test, y_train, y_test = train_test_split(
    X_sj, y_sj, test_size=0.2, random_state=862
)
model_sj = xgb.XGBRegressor(n_estimators=300)

param_grid_sj = {
    'colsample_bytree': [0.9],
    'learning_rate':    [0.121],
    'max_depth':        [7],
    'alpha':            [11.35]
}
grid_sj = GridSearchCV(model_sj, param_grid_sj, scoring='r2', cv=5, n_jobs=-1)
grid_sj.fit(X_train, y_train)
best_sj = grid_sj.best_estimator_

print("SJ Train R2:", metrics.r2_score(y_train, best_sj.predict(X_train)))
print("SJ Test  R2:", metrics.r2_score(y_test,  best_sj.predict(X_test)))

joblib.dump(best_sj, os.path.join(MODEL_DIR, 'best_sj.pkl'))

RP Train R2: 0.9420456443903478
RP Test  R2: 0.8250562477129679
SJ Train R2: 0.9587047068120595
SJ Test  R2: 0.8692201754862173


['models\\best_sj.pkl']

In [3]:
#run_ga_target1-H2O LiCl.py
import os
import random
from copy import deepcopy

import joblib
import numpy as np
import pandas as pd

# =====================================================================
# 0. Load trained models
# =====================================================================
MODEL_DIR = 'models'
best_rp = joblib.load(os.path.join(MODEL_DIR, 'best_rp.pkl'))
best_sj = joblib.load(os.path.join(MODEL_DIR, 'best_sj.pkl'))

# =====================================================================
# 1. Random seed
# =====================================================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# =====================================================================
# 2. Target intervals
# =====================================================================
rp_low, rp_high = 0.45, 0.465
sj_low, sj_high = 140, 1000

# =====================================================================
# 3. Fixed parameters
# =====================================================================
X0    = 80
SJ_X1 = 0.75

RP_WEIGHT = 100
SJ_WEIGHT = 1

# =====================================================================
# 4. Search space
# =====================================================================
PIP_list = np.unique(np.round(np.arange(0.05, 0.501, 0.01), 2))
TMC_list = np.unique(np.round(np.arange(0.01, 0.251, 0.01), 2))

N_PIP = len(PIP_list)
N_TMC = len(TMC_list)

print(f"PIP grid size: {N_PIP}")
print(f"TMC grid size: {N_TMC}")
print(f"Total combinations: {N_PIP * N_TMC}")

# =====================================================================
# 5. GA hyperparameters
# =====================================================================
POP_SIZE        = 80
N_GENERATIONS   = 200
ELITE_SIZE      = 8
TOURNAMENT_SIZE = 4

CROSSOVER_RATE      = 0.85
MUTATION_RATE       = 0.30
RESET_MUTATION_RATE = 0.10

MUTATION_STEP_PIP = 3
MUTATION_STEP_TMC = 3

EARLY_STOP_GENERATIONS = 500


# =====================================================================
# 6. Helper functions
# =====================================================================
def interval_error(value, low, high):
    if value < low:
        return low - value
    elif value > high:
        return value - high
    else:
        return 0.0


def safe_predict(model, X):
    return float(np.ravel(model.predict(X))[0])


# =====================================================================
# 7. Gene encoding / decoding (gene = [PIP_index, TMC_index])
# =====================================================================
def random_gene():
    return [
        random.randint(0, N_PIP - 1),
        random.randint(0, N_TMC - 1)
    ]


def decode_gene(gene):
    pip_idx = int(np.clip(gene[0], 0, N_PIP - 1))
    tmc_idx = int(np.clip(gene[1], 0, N_TMC - 1))

    pip_val   = float(PIP_list[pip_idx])
    tmc_val   = float(TMC_list[tmc_idx])
    ratio_val = pip_val / tmc_val

    return pip_idx, tmc_idx, pip_val, tmc_val, ratio_val


# =====================================================================
# 8. Fitness function
# =====================================================================
eval_cache = {}


def evaluate_gene(gene):
    pip_idx, tmc_idx, pip_val, tmc_val, ratio_val = decode_gene(gene)

    cache_key = (pip_idx, tmc_idx)
    if cache_key in eval_cache:
        return deepcopy(eval_cache[cache_key])

    # RP prediction: input = [PIP, TMC, PIP/TMC]
    X_rp_input = np.array([[pip_val, tmc_val, ratio_val]], dtype=float)
    y_rp_pred  = safe_predict(best_rp, X_rp_input)
    err_rp     = interval_error(y_rp_pred, rp_low, rp_high)

    # SJ prediction: input = [X0, SJ_X1, PIP, TMC, PIP/TMC]
    X_sj_input = np.array([[X0, SJ_X1, pip_val, tmc_val, ratio_val]], dtype=float)
    y_sj_pred  = safe_predict(best_sj, X_sj_input)
    err_sj     = interval_error(y_sj_pred, sj_low, sj_high)

    total_err = RP_WEIGHT * err_rp + SJ_WEIGHT * err_sj
    if not np.isfinite(total_err):
        total_err = 1e18

    record = {
        "PIP_idx":       pip_idx,
        "TMC_idx":       tmc_idx,
        "PIP_X1":        pip_val,
        "TMC_X2":        tmc_val,
        "PIP_TMC_ratio": ratio_val,
        "y_rp":          y_rp_pred,
        "err_rp":        err_rp,
        "y_sj":          y_sj_pred,
        "err_sj":        err_sj,
        "RP_WEIGHT":     RP_WEIGHT,
        "SJ_WEIGHT":     SJ_WEIGHT,
        "total_err":     total_err,
        "rp_in_range":   int(rp_low <= y_rp_pred <= rp_high),
        "sj_in_range":   int(sj_low <= y_sj_pred <= sj_high),
        "both_in_range": int((rp_low <= y_rp_pred <= rp_high)
                             and (sj_low <= y_sj_pred <= sj_high))
    }

    eval_cache[cache_key] = deepcopy(record)
    return record


# =====================================================================
# 9. GA operators
# =====================================================================
def tournament_select(evaluated_population):
    candidates = random.sample(evaluated_population, TOURNAMENT_SIZE)
    candidates = sorted(candidates, key=lambda x: x[1]["total_err"])
    return deepcopy(candidates[0][0])


def crossover(parent1, parent2):
    if random.random() > CROSSOVER_RATE:
        return deepcopy(parent1)
    return [
        parent1[0] if random.random() < 0.5 else parent2[0],
        parent1[1] if random.random() < 0.5 else parent2[1]
    ]


def mutate(gene):
    new_gene = deepcopy(gene)
    if random.random() < MUTATION_RATE:
        # PIP mutation
        if random.random() < 0.5:
            if random.random() < RESET_MUTATION_RATE:
                new_gene[0] = random.randint(0, N_PIP - 1)
            else:
                step = random.randint(-MUTATION_STEP_PIP, MUTATION_STEP_PIP)
                new_gene[0] = int(np.clip(new_gene[0] + step, 0, N_PIP - 1))
        # TMC mutation
        if random.random() < 0.5:
            if random.random() < RESET_MUTATION_RATE:
                new_gene[1] = random.randint(0, N_TMC - 1)
            else:
                step = random.randint(-MUTATION_STEP_TMC, MUTATION_STEP_TMC)
                new_gene[1] = int(np.clip(new_gene[1] + step, 0, N_TMC - 1))
    return new_gene


# =====================================================================
# 10. Initialize population
# =====================================================================
population = [random_gene() for _ in range(POP_SIZE)]

best_global_gene   = None
best_global_record = None
best_global_score  = np.inf

history          = []
all_records      = {}
no_improve_count = 0

# =====================================================================
# 11. GA main loop
# =====================================================================
for gen in range(1, N_GENERATIONS + 1):

    evaluated_population = []
    for gene in population:
        record = evaluate_gene(gene)
        evaluated_population.append((deepcopy(gene), record))
        all_records[(record["PIP_idx"], record["TMC_idx"])] = deepcopy(record)

    evaluated_population.sort(key=lambda x: x[1]["total_err"])

    best_gene_this_gen, best_record_this_gen = evaluated_population[0]
    best_score_this_gen = best_record_this_gen["total_err"]
    mean_score_this_gen = np.mean([x[1]["total_err"] for x in evaluated_population])

    if best_score_this_gen < best_global_score:
        best_global_score  = best_score_this_gen
        best_global_gene   = deepcopy(best_gene_this_gen)
        best_global_record = deepcopy(best_record_this_gen)
        no_improve_count   = 0
    else:
        no_improve_count += 1

    history.append({
        "generation":              gen,
        "best_total_err_this_gen": best_score_this_gen,
        "global_best_total_err":   best_global_score,
        "mean_total_err":          mean_score_this_gen,
        "best_PIP_X1":             best_record_this_gen["PIP_X1"],
        "best_TMC_X2":             best_record_this_gen["TMC_X2"],
        "best_ratio":              best_record_this_gen["PIP_TMC_ratio"],
        "best_y_rp":               best_record_this_gen["y_rp"],
        "best_err_rp":             best_record_this_gen["err_rp"],
        "best_y_sj":               best_record_this_gen["y_sj"],
        "best_err_sj":             best_record_this_gen["err_sj"],
        "both_in_range":           best_record_this_gen["both_in_range"]
    })

    if gen == 1 or gen % 10 == 0 or best_record_this_gen["both_in_range"] == 1:
        print(
            f"Gen {gen:03d} | "
            f"Best total_err = {best_score_this_gen:.6f} | "
            f"Global best = {best_global_score:.6f} | "
            f"PIP = {best_record_this_gen['PIP_X1']:.3f} | "
            f"TMC = {best_record_this_gen['TMC_X2']:.3f} | "
            f"Ratio = {best_record_this_gen['PIP_TMC_ratio']:.4f} | "
            f"RP = {best_record_this_gen['y_rp']:.6f} | "
            f"SJ = {best_record_this_gen['y_sj']:.6f}"
        )

    # Early stop when a feasible solution is found
    if best_global_record is not None and best_global_record["total_err"] <= 1e-12:
        print(f"\nFeasible solution (total_err = 0) found at generation {gen}. Stopping early.")
        break

    if no_improve_count >= EARLY_STOP_GENERATIONS:
        print(f"\nNo improvement for {EARLY_STOP_GENERATIONS} consecutive generations. Stopping early.")
        break

    # Build next generation
    next_population = []
    for elite_gene, _ in evaluated_population[:ELITE_SIZE]:
        next_population.append(deepcopy(elite_gene))
    while len(next_population) < POP_SIZE:
        parent1 = tournament_select(evaluated_population)
        parent2 = tournament_select(evaluated_population)
        child   = crossover(parent1, parent2)
        child   = mutate(child)
        next_population.append(child)
    population = next_population


# =====================================================================
# 12. Collect results
# =====================================================================
df_ga        = pd.DataFrame(list(all_records.values()))
df_ga_sorted = df_ga.sort_values("total_err").reset_index(drop=True)
history_df   = pd.DataFrame(history)

print("\n===== Best GA candidate =====")
print(pd.Series(best_global_record).to_string())

print("\n===== Top-20 candidates by total_err =====")
print(df_ga_sorted.head(20).to_string(index=False))

# =====================================================================
# 13. Export to Excel
# =====================================================================
output_dir = 'results'
os.makedirs(output_dir, exist_ok=True)

output_excel = os.path.join(output_dir, 'GA_PIP_TMC_total_err_results_target1.xlsx')
with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    df_ga_sorted.to_excel(writer, sheet_name='GA_candidates', index=False)
    history_df.to_excel(writer,   sheet_name='GA_history',    index=False)

print(f"\nSaved Excel: {output_excel}")

# =====================================================================
# 14. Deduplicate by predicted values
#     Sort by total_err first, then by PIP_X1 in descending order
# =====================================================================
df_unique_pred = df_ga_sorted.copy()

df_unique_pred["y_rp_round"]      = df_unique_pred["y_rp"].round(6)
df_unique_pred["y_sj_round"]      = df_unique_pred["y_sj"].round(6)
df_unique_pred["total_err_round"] = df_unique_pred["total_err"].round(8)

PIP_CENTER = 0.33
TMC_CENTER = 0.18

df_unique_pred["center_distance"] = np.sqrt(
    (df_unique_pred["PIP_X1"] - PIP_CENTER) ** 2 +
    (df_unique_pred["TMC_X2"] - TMC_CENTER) ** 2
)

# Within the same predicted-value group, keep the one with the smallest total_err
# and (as a tiebreaker) closest to the recipe center.
df_unique_pred = (
    df_unique_pred
    .sort_values(["total_err", "center_distance"])
    .drop_duplicates(
        subset=["y_rp_round", "y_sj_round", "total_err_round"],
        keep="first"
    )
    .reset_index(drop=True)
)

# Final ordering: by total_err ascending, then PIP_X1 descending
df_unique_pred = df_unique_pred.sort_values(
    by=["total_err", "PIP_X1"],
    ascending=[True, False]
).reset_index(drop=True)

df_unique_pred_clean = df_unique_pred.drop(
    columns=["y_rp_round", "y_sj_round", "total_err_round", "center_distance"]
)

print("\n===== Unique candidates after deduplication by prediction =====")
print(df_unique_pred_clean.head(20).to_string(index=False))

output_unique_excel = os.path.join(output_dir, 'GA_unique_prediction_candidates_target1.xlsx')
df_unique_pred_clean.to_excel(output_unique_excel, index=False)

print(f"\nSaved: {output_unique_excel}")

PIP grid size: 46
TMC grid size: 25
Total combinations: 1150
Gen 001 | Best total_err = 0.513521 | Global best = 0.513521 | PIP = 0.330 | TMC = 0.190 | Ratio = 1.7368 | RP = 0.470135 | SJ = 153.179688
Gen 010 | Best total_err = 0.207934 | Global best = 0.207934 | PIP = 0.310 | TMC = 0.190 | Ratio = 1.6316 | RP = 0.467079 | SJ = 140.468826
Gen 020 | Best total_err = 0.207934 | Global best = 0.207934 | PIP = 0.310 | TMC = 0.190 | Ratio = 1.6316 | RP = 0.467079 | SJ = 140.468826
Gen 030 | Best total_err = 0.207934 | Global best = 0.207934 | PIP = 0.310 | TMC = 0.190 | Ratio = 1.6316 | RP = 0.467079 | SJ = 140.468826
Gen 040 | Best total_err = 0.207934 | Global best = 0.207934 | PIP = 0.310 | TMC = 0.190 | Ratio = 1.6316 | RP = 0.467079 | SJ = 140.468826
Gen 050 | Best total_err = 0.207934 | Global best = 0.207934 | PIP = 0.310 | TMC = 0.190 | Ratio = 1.6316 | RP = 0.467079 | SJ = 140.468826
Gen 060 | Best total_err = 0.207934 | Global best = 0.207934 | PIP = 0.310 | TMC = 0.190 | Ratio = 

In [4]:
#run_ga_target2-H2O MgCl2.py
import os
import random
from copy import deepcopy

import joblib
import numpy as np
import pandas as pd

# =====================================================================
# 0. Load trained models
# =====================================================================
MODEL_DIR = 'models'
best_rp = joblib.load(os.path.join(MODEL_DIR, 'best_rp.pkl'))
best_sj = joblib.load(os.path.join(MODEL_DIR, 'best_sj.pkl'))

# =====================================================================
# 1. Random seed
# =====================================================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# =====================================================================
# 2. Target intervals
# =====================================================================
rp_low, rp_high = 0.52, 0.54
sj_low, sj_high = 30, 50

# =====================================================================
# 3. Fixed parameters
# =====================================================================
X0    = 80
SJ_X1 = 0.75

RP_WEIGHT = 100
SJ_WEIGHT = 1

# =====================================================================
# 4. Search space
# =====================================================================
PIP_list = np.unique(np.round(np.arange(0.05, 0.501, 0.01), 2))
TMC_list = np.unique(np.round(np.arange(0.01, 0.251, 0.01), 2))

N_PIP = len(PIP_list)
N_TMC = len(TMC_list)

print(f"PIP grid size: {N_PIP}")
print(f"TMC grid size: {N_TMC}")
print(f"Total combinations: {N_PIP * N_TMC}")

# =====================================================================
# 5. GA hyperparameters
# =====================================================================
POP_SIZE        = 80
N_GENERATIONS   = 200
ELITE_SIZE      = 8
TOURNAMENT_SIZE = 4

CROSSOVER_RATE      = 0.85
MUTATION_RATE       = 0.30
RESET_MUTATION_RATE = 0.10

MUTATION_STEP_PIP = 3
MUTATION_STEP_TMC = 3

EARLY_STOP_GENERATIONS = 500


# =====================================================================
# 6. Helper functions
# =====================================================================
def interval_error(value, low, high):
    if value < low:
        return low - value
    elif value > high:
        return value - high
    else:
        return 0.0


def safe_predict(model, X):
    return float(np.ravel(model.predict(X))[0])


# =====================================================================
# 7. Gene encoding / decoding (gene = [PIP_index, TMC_index])
# =====================================================================
def random_gene():
    return [
        random.randint(0, N_PIP - 1),
        random.randint(0, N_TMC - 1)
    ]


def decode_gene(gene):
    pip_idx = int(np.clip(gene[0], 0, N_PIP - 1))
    tmc_idx = int(np.clip(gene[1], 0, N_TMC - 1))

    pip_val   = float(PIP_list[pip_idx])
    tmc_val   = float(TMC_list[tmc_idx])
    ratio_val = pip_val / tmc_val

    return pip_idx, tmc_idx, pip_val, tmc_val, ratio_val


# =====================================================================
# 8. Fitness function
# =====================================================================
eval_cache = {}


def evaluate_gene(gene):
    pip_idx, tmc_idx, pip_val, tmc_val, ratio_val = decode_gene(gene)

    cache_key = (pip_idx, tmc_idx)
    if cache_key in eval_cache:
        return deepcopy(eval_cache[cache_key])

    X_rp_input = np.array([[pip_val, tmc_val, ratio_val]], dtype=float)
    y_rp_pred  = safe_predict(best_rp, X_rp_input)
    err_rp     = interval_error(y_rp_pred, rp_low, rp_high)

    X_sj_input = np.array([[X0, SJ_X1, pip_val, tmc_val, ratio_val]], dtype=float)
    y_sj_pred  = safe_predict(best_sj, X_sj_input)
    err_sj     = interval_error(y_sj_pred, sj_low, sj_high)

    total_err = RP_WEIGHT * err_rp + SJ_WEIGHT * err_sj
    if not np.isfinite(total_err):
        total_err = 1e18

    record = {
        "PIP_idx":       pip_idx,
        "TMC_idx":       tmc_idx,
        "PIP_X1":        pip_val,
        "TMC_X2":        tmc_val,
        "PIP_TMC_ratio": ratio_val,
        "y_rp":          y_rp_pred,
        "err_rp":        err_rp,
        "y_sj":          y_sj_pred,
        "err_sj":        err_sj,
        "RP_WEIGHT":     RP_WEIGHT,
        "SJ_WEIGHT":     SJ_WEIGHT,
        "total_err":     total_err,
        "rp_in_range":   int(rp_low <= y_rp_pred <= rp_high),
        "sj_in_range":   int(sj_low <= y_sj_pred <= sj_high),
        "both_in_range": int((rp_low <= y_rp_pred <= rp_high)
                             and (sj_low <= y_sj_pred <= sj_high))
    }

    eval_cache[cache_key] = deepcopy(record)
    return record


# =====================================================================
# 9. GA operators
# =====================================================================
def tournament_select(evaluated_population):
    candidates = random.sample(evaluated_population, TOURNAMENT_SIZE)
    candidates = sorted(candidates, key=lambda x: x[1]["total_err"])
    return deepcopy(candidates[0][0])


def crossover(parent1, parent2):
    if random.random() > CROSSOVER_RATE:
        return deepcopy(parent1)
    return [
        parent1[0] if random.random() < 0.5 else parent2[0],
        parent1[1] if random.random() < 0.5 else parent2[1]
    ]


def mutate(gene):
    new_gene = deepcopy(gene)
    if random.random() < MUTATION_RATE:
        if random.random() < 0.5:
            if random.random() < RESET_MUTATION_RATE:
                new_gene[0] = random.randint(0, N_PIP - 1)
            else:
                step = random.randint(-MUTATION_STEP_PIP, MUTATION_STEP_PIP)
                new_gene[0] = int(np.clip(new_gene[0] + step, 0, N_PIP - 1))
        if random.random() < 0.5:
            if random.random() < RESET_MUTATION_RATE:
                new_gene[1] = random.randint(0, N_TMC - 1)
            else:
                step = random.randint(-MUTATION_STEP_TMC, MUTATION_STEP_TMC)
                new_gene[1] = int(np.clip(new_gene[1] + step, 0, N_TMC - 1))
    return new_gene


# =====================================================================
# 10. Initialize population
# =====================================================================
population = [random_gene() for _ in range(POP_SIZE)]

best_global_gene   = None
best_global_record = None
best_global_score  = np.inf

history          = []
all_records      = {}
no_improve_count = 0

# =====================================================================
# 11. GA main loop
# =====================================================================
for gen in range(1, N_GENERATIONS + 1):

    evaluated_population = []
    for gene in population:
        record = evaluate_gene(gene)
        evaluated_population.append((deepcopy(gene), record))
        all_records[(record["PIP_idx"], record["TMC_idx"])] = deepcopy(record)

    evaluated_population.sort(key=lambda x: x[1]["total_err"])

    best_gene_this_gen, best_record_this_gen = evaluated_population[0]
    best_score_this_gen = best_record_this_gen["total_err"]
    mean_score_this_gen = np.mean([x[1]["total_err"] for x in evaluated_population])

    if best_score_this_gen < best_global_score:
        best_global_score  = best_score_this_gen
        best_global_gene   = deepcopy(best_gene_this_gen)
        best_global_record = deepcopy(best_record_this_gen)
        no_improve_count   = 0
    else:
        no_improve_count += 1

    history.append({
        "generation":              gen,
        "best_total_err_this_gen": best_score_this_gen,
        "global_best_total_err":   best_global_score,
        "mean_total_err":          mean_score_this_gen,
        "best_PIP_X1":             best_record_this_gen["PIP_X1"],
        "best_TMC_X2":             best_record_this_gen["TMC_X2"],
        "best_ratio":              best_record_this_gen["PIP_TMC_ratio"],
        "best_y_rp":               best_record_this_gen["y_rp"],
        "best_err_rp":             best_record_this_gen["err_rp"],
        "best_y_sj":               best_record_this_gen["y_sj"],
        "best_err_sj":             best_record_this_gen["err_sj"],
        "both_in_range":           best_record_this_gen["both_in_range"]
    })

    if gen == 1 or gen % 10 == 0 or best_record_this_gen["both_in_range"] == 1:
        print(
            f"Gen {gen:03d} | "
            f"Best total_err = {best_score_this_gen:.6f} | "
            f"Global best = {best_global_score:.6f} | "
            f"PIP = {best_record_this_gen['PIP_X1']:.3f} | "
            f"TMC = {best_record_this_gen['TMC_X2']:.3f} | "
            f"Ratio = {best_record_this_gen['PIP_TMC_ratio']:.4f} | "
            f"RP = {best_record_this_gen['y_rp']:.6f} | "
            f"SJ = {best_record_this_gen['y_sj']:.6f}"
        )

    if no_improve_count >= EARLY_STOP_GENERATIONS:
        print(f"\nNo improvement for {EARLY_STOP_GENERATIONS} consecutive generations. Stopping early.")
        break

    next_population = []
    for elite_gene, _ in evaluated_population[:ELITE_SIZE]:
        next_population.append(deepcopy(elite_gene))
    while len(next_population) < POP_SIZE:
        parent1 = tournament_select(evaluated_population)
        parent2 = tournament_select(evaluated_population)
        child   = crossover(parent1, parent2)
        child   = mutate(child)
        next_population.append(child)
    population = next_population


# =====================================================================
# 12. Collect results
# =====================================================================
df_ga        = pd.DataFrame(list(all_records.values()))
df_ga_sorted = df_ga.sort_values("total_err").reset_index(drop=True)
history_df   = pd.DataFrame(history)

print("\n===== Best GA candidate =====")
print(pd.Series(best_global_record).to_string())

print("\n===== Top-20 candidates by total_err =====")
print(df_ga_sorted.head(20).to_string(index=False))

# =====================================================================
# 13. Export to Excel
# =====================================================================
output_dir = 'results'
os.makedirs(output_dir, exist_ok=True)

output_excel = os.path.join(output_dir, 'GA_PIP_TMC_total_err_results_target2.xlsx')
with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    df_ga_sorted.to_excel(writer, sheet_name='GA_candidates', index=False)
    history_df.to_excel(writer,   sheet_name='GA_history',    index=False)

print(f"\nSaved Excel: {output_excel}")

# =====================================================================
# 14. Deduplicate by predicted values
#     Sort by total_err first, then by PIP_X1 in descending order
# =====================================================================
df_unique_pred = df_ga_sorted.copy()

df_unique_pred["y_rp_round"]      = df_unique_pred["y_rp"].round(6)
df_unique_pred["y_sj_round"]      = df_unique_pred["y_sj"].round(6)
df_unique_pred["total_err_round"] = df_unique_pred["total_err"].round(8)

PIP_CENTER = 0.33
TMC_CENTER = 0.18

df_unique_pred["center_distance"] = np.sqrt(
    (df_unique_pred["PIP_X1"] - PIP_CENTER) ** 2 +
    (df_unique_pred["TMC_X2"] - TMC_CENTER) ** 2
)

df_unique_pred = (
    df_unique_pred
    .sort_values(["total_err", "center_distance"])
    .drop_duplicates(
        subset=["y_rp_round", "y_sj_round", "total_err_round"],
        keep="first"
    )
    .reset_index(drop=True)
)

# Final ordering: by total_err ascending, then PIP_X1 descending
df_unique_pred = df_unique_pred.sort_values(
    by=["total_err", "PIP_X1"],
    ascending=[True, False]
).reset_index(drop=True)

df_unique_pred_clean = df_unique_pred.drop(
    columns=["y_rp_round", "y_sj_round", "total_err_round", "center_distance"]
)

print("\n===== Unique candidates after deduplication by prediction =====")
print(df_unique_pred_clean.head(20).to_string(index=False))

output_unique_excel = os.path.join(output_dir, 'GA_unique_prediction_candidates_target2.xlsx')
df_unique_pred_clean.to_excel(output_unique_excel, index=False)

print(f"\nSaved: {output_unique_excel}")

PIP grid size: 46
TMC grid size: 25
Total combinations: 1150
Gen 001 | Best total_err = 0.000000 | Global best = 0.000000 | PIP = 0.080 | TMC = 0.040 | Ratio = 2.0000 | RP = 0.524288 | SJ = 43.023666
Gen 002 | Best total_err = 0.000000 | Global best = 0.000000 | PIP = 0.080 | TMC = 0.040 | Ratio = 2.0000 | RP = 0.524288 | SJ = 43.023666
Gen 003 | Best total_err = 0.000000 | Global best = 0.000000 | PIP = 0.080 | TMC = 0.040 | Ratio = 2.0000 | RP = 0.524288 | SJ = 43.023666
Gen 004 | Best total_err = 0.000000 | Global best = 0.000000 | PIP = 0.080 | TMC = 0.040 | Ratio = 2.0000 | RP = 0.524288 | SJ = 43.023666
Gen 005 | Best total_err = 0.000000 | Global best = 0.000000 | PIP = 0.080 | TMC = 0.040 | Ratio = 2.0000 | RP = 0.524288 | SJ = 43.023666
Gen 006 | Best total_err = 0.000000 | Global best = 0.000000 | PIP = 0.080 | TMC = 0.040 | Ratio = 2.0000 | RP = 0.524288 | SJ = 43.023666
Gen 007 | Best total_err = 0.000000 | Global best = 0.000000 | PIP = 0.080 | TMC = 0.040 | Ratio = 2.0000